In [ ]:
Q1.
(a)None
(b)'2026-05-06'
(c)['2026-06-06', '2026-05-18']
(d)[('2026', '05', '06'), ('2026', '05', '18')]
(e)['2026-05-06', '2026-05-18']
re.findall은 패턴 내에 소괄호 ()로 지정된 캡처 그룹이 있으면 전체 매치 대신 그룹에 매칭된 부분만 추출한다.
(d)는 3개의 캡처 그룹이 있어 매칭된 부분 문자열들을 튜플로 묶어 반환한다.
(e)는 (?:...)를 사용한 비캡처 그룹이므로 캡처를 건너뛰고 공백을 포함한 전체 매칭 문자열을 리스트로 반환한다.

In [ ]:

Q2.
(a) "[T]!"
(b) "[T]안녕[T] [T]세상[T]!"
(c) "[T]안녕[T] [T]세상[T]!"
(d) "수강생 <30>명, 조교 <3>명"
(e) "수강생 <\x18>명, 조교 <\x18>명"
i)(a)의 .+는 매칭을 최대한 길게 하려는 탐욕적 속성을 가져 첫 <b>부터 마지막 </i>까지를 통째로 치환하지만, 
(b)의 .+?는 가장 짧게 매칭하려는 게으른 속성을 가져 각 HTML 태그를 개별적으로 찾아 치환하기 때문이다.
ii)(d)는 원시 문자열을 사용하여 \1이 첫 번째 캡처 그룹을 가리키는 백레퍼런스로 올바르게 해석되지만, 
(e)는 일반 문자열이어서 \1이 8진수 이스케이프 문자(\x18)로 먼저 해석되어 치환되기 때문이다.

In [ ]:
import re
from collections import Counter
from typing import Any

URL_PATTERN: re.Pattern[str] = re.compile(r"https?://\S+")
HTML_PATTERN: re.Pattern[str] = re.compile(r"<[^>]+>")
EMAIL_PATTERN: re.Pattern[str] = re.compile(
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
)
PHONE_PATTERN: re.Pattern[str] = re.compile(r"\d{2,4}-\d{3,4}-\d{4}")
MENTION_HASHTAG_PATTERN: re.Pattern[str] = re.compile(r"[@#]\w+")
KOREAN_JAMO_PATTERN: re.Pattern[str] = re.compile(r"[ㄱ-ㅎㅏ-ㅣ]+")
WHITESPACE_PATTERN: re.Pattern[str] = re.compile(r"\s+")
HASHTAG_EXTRACT_PATTERN: re.Pattern[str] = re.compile(r"#(\w+)")


def clean_post(post: str) -> str:
    post = URL_PATTERN.sub(" ", post)
    post = HTML_PATTERN.sub("", post)
    post = EMAIL_PATTERN.sub("[이메일]", post)
    post = PHONE_PATTERN.sub("[전화]", post)
    post = MENTION_HASHTAG_PATTERN.sub(" ", post)
    post = KOREAN_JAMO_PATTERN.sub("", post)
    post = WHITESPACE_PATTERN.sub(" ", post).strip()
    return post


def extract_hashtags(post: str) -> list[str]:
    return HASHTAG_EXTRACT_PATTERN.findall(post)


def analyze_posts(posts: list[str]) -> dict[str, Any]:
    posts_n: int = len(posts)
    cleaned_posts: list[str] = []
    total_masked_count: int = 0
    all_hashtags: list[str] = []

    for post in posts:
        all_hashtags.extend(extract_hashtags(post))

        p: str = URL_PATTERN.sub(" ", post)
        p = HTML_PATTERN.sub("", p)

        p, email_cnt = EMAIL_PATTERN.subn("[이메일]", p)
        p, phone_cnt = PHONE_PATTERN.subn("[전화]", p)
        total_masked_count += email_cnt + phone_cnt

        p = MENTION_HASHTAG_PATTERN.sub(" ", p)
        p = KOREAN_JAMO_PATTERN.sub("", p)
        p = WHITESPACE_PATTERN.sub(" ", p).strip()

        cleaned_posts.append(p)

    if posts_n > 0:
        avg_length: float = round(
            sum(len(p) for p in cleaned_posts) / posts_n, 2
        )
    else:
        avg_length = 0.0

    hashtag_counts: dict[str, int] = dict(Counter(all_hashtags).most_common())

    return {
        "posts_n": posts_n,
        "avg_length_after_clean": avg_length,
        "hashtag_counts": hashtag_counts,
        "masked_count": total_masked_count,
    }


if __name__ == "__main__":
    posts_data: list[str] = [
        "오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ",
        "자료: https://etl.snu.ac.kr/lec17",
        "@lee @park 팀플 어디서 모이지ㅠㅠㅠㅠㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ",
        "<b>중요</b>: 다음 시험 범위는 1-15강.",
        "문의는 mam3b@snu.ac.kr (010-1234-5678)로!",
        "여러 공백과\n\n\n줄바꿈이 많은 텍스트",
        "ㅋㅋㅋ #파이썬 진짜 좋다 #추천 https://snu.ac.kr",
    ]

    print("--- (1) 각 게시물에 대한 clean_post 결과 ---")
    for i, post in enumerate(posts_data, 1):
        print(f"Post {i}: {clean_post(post)}")

    print("\n--- (2) analyze_posts(posts) 결과 ---")
    import pprint

    pprint.pprint(analyze_posts(posts_data))

In [ ]:
--- (1) 각 게시물에 대한 clean_post 결과 ---
Post 1: 오늘 수업 진짜 재밌었음!! 감사
Post 2: 자료:
Post 3: 팀플 어디서 모이지 카톡
Post 4: 중요: 다음 시험 범위는 1-15강.
Post 5: 문의는 [이메일] ([전화])로!
Post 6: 여러 공백과 줄바꿈이 많은 텍스트
Post 7: 진짜 좋다

--- (2) analyze_posts(posts) 결과 ---
{'avg_length_after_clean': 15.14,
 'hashtag_counts': {'파이썬': 2, 'DCCP2026': 1, '추천': 1, '팀플': 1},
 'masked_count': 2,
 'posts_n': 7}

In [ ]:
만약 단계 3(개인정보 마스킹)과 단계 4(멘션 및 해시태그 치환)의 순서를 바꾸게 되면, 
이메일 주소 내에 포함된 @ 기호가 단계 4의 멘션 패턴(@\w+)에 의해 먼저 매칭되어 공백으로 치환·파괴된다. 
이로 인해 이메일 문자열의 고유 구조가 유실되므로, 
이후 단계 3에서 이메일을 정상적으로 탐지하여 마스킹하는 것이 불가능해진다. 
따라서 주어진 파이프라인의 순서를 엄격히 준수해야 한다.